In [5]:
"""
CitySync Command — V7 Final Production Model
=============================================
LightGBM Quantile Regression + Soft Severity + Urgency Score

Architecture:
  Layer 1 — Quantile Regression (q10 / q50 / q90)
  Layer 2 — Soft Severity Classifier (asymmetric band-aware, 3-tier confidence)
  Layer 3 — Urgency Score (composite 0–100, percentage-based, 4 tiers)

Confidence logic (asymmetric, p50-upward focus):
  High     : p10, p50, p90 all same bucket
  Moderate : p50 and p90 same bucket (upper half contained)
  Low      : p50 and p90 different buckets (genuine uncertainty)
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, f1_score, confusion_matrix
from sklearn.cluster import KMeans
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ─────────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────────
DATA_PATH        = '/kaggle/input/datasets/dg8763/theme2/theme2.csv'
OUTPUT_PKL       = 'citysync_v7.pkl'
FIG_IMPORTANCE   = 'fig_v7_importance.png'
FIG_CONFUSION    = 'fig_v7_confusion.png'
FIG_CALIBRATION  = 'fig_v7_calibration.png'

RANDOM_SEED      = 42
MAX_DURATION_HRS = 72
MIN_CAT_FREQ     = 5
VB_JUNC_FREQ     = 5
N_SPATIAL_ZONES  = 25
OPTUNA_TRIALS    = 60
CV_SPLITS        = 4
TRAIN_RATIO      = 0.80
BENGALURU_LAT    = 12.9716
BENGALURU_LNG    = 77.5946

# Data-grounded peak hours (from actual incident distribution)
# Top hours by count: 21, 20, 5, 6, 4, 19, 22 — NOT commuter [8,9,17,18]
NIGHT_HOURS      = [19, 20, 21, 22, 23, 0, 1, 2, 3]
PREDAWN_HOURS    = [4, 5, 6, 7]

BUCKET_BOUNDS    = [0, 1, 6, 24, 72]
BUCKET_LABELS    = ['Short', 'Medium', 'Long', 'Extended']
BUCKET_COLOURS   = ['#00c48c', '#f5a623', '#ff6b35', '#ff3b55']
URGENCY_TIERS    = {'CRITICAL': 80, 'HIGH': 55, 'MODERATE': 30, 'LOW': 0}

DARK_BG  = '#090d14'
SURFACE  = '#0f1520'
BORDER   = '#1e2d42'
ACCENT   = '#00e5ff'
MUTED    = '#5a6a82'
TEXT     = '#e8edf5'


# ─────────────────────────────────────────────────
# 1. LOAD & TARGET PIPELINE
# ─────────────────────────────────────────────────
print("=" * 60)
print("  CITYSYNC COMMAND — V7 FINAL MODEL")
print("=" * 60)
print("\n[1] Loading and resolving target variable...")

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['start_datetime']).copy()

for col in ['start_datetime', 'end_datetime', 'resolved_datetime', 'closed_datetime']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

# Priority chain: end_datetime (planned) → resolved → closed
df['end_time'] = (
    df['end_datetime']
    .combine_first(df['resolved_datetime'])
    .combine_first(df['closed_datetime'])
)
df = df.dropna(subset=['end_time']).copy()
df['duration_hours'] = (
    (df['end_time'] - df['start_datetime']).dt.total_seconds() / 3600
)

# Cap at 72 hrs: excludes pot_holes (~9d median), debris (~60d) —
# chronic infrastructure failures, not dispatchable traffic events
df = df[(df['duration_hours'] > 0) & (df['duration_hours'] <= MAX_DURATION_HRS)].copy()
df = df.sort_values('start_datetime').reset_index(drop=True)

print(f"    Records : {len(df)}")
print(f"    Range   : {df['duration_hours'].min():.2f} – {df['duration_hours'].max():.2f} hrs")
print(f"    Median  : {df['duration_hours'].median():.2f} hrs  "
      f"(~{df['duration_hours'].median()*60:.0f} min)")
print(f"    Dates   : {df['start_datetime'].min().date()} → "
      f"{df['start_datetime'].max().date()}")


# ─────────────────────────────────────────────────
# 2. LEAKAGE-FREE ROLLING CORRIDOR COUNTS
#    Replaces active_system_load which leaked via end_time.
#    Uses only start_datetime — fully computable at inference time.
# ─────────────────────────────────────────────────
print("\n[2] Computing leakage-free rolling corridor counts...")

starts    = df['start_datetime'].values.astype(np.int64)
corridors = df['corridor'].values
ns_1h     = int(1 * 3600 * 1e9)
ns_3h     = int(3 * 3600 * 1e9)
ns_6h     = int(6 * 3600 * 1e9)

c1 = np.zeros(len(df), dtype=np.int32)
c3 = np.zeros(len(df), dtype=np.int32)
c6 = np.zeros(len(df), dtype=np.int32)

for i in range(len(df)):
    t  = starts[i]
    co = corridors[i]
    lo6 = np.searchsorted(starts, t - ns_6h, side='left')
    lo3 = np.searchsorted(starts, t - ns_3h, side='left')
    lo1 = np.searchsorted(starts, t - ns_1h, side='left')
    hi  = i  # strict: exclude self, no end_time referenced
    if hi > lo6:
        win6 = corridors[lo6:hi]
        c6[i] = (win6 == co).sum()
        if hi > lo3:
            c3[i] = (corridors[lo3:hi] == co).sum()
        if hi > lo1:
            c1[i] = (corridors[lo1:hi] == co).sum()

df['events_1h'] = c1
df['events_3h'] = c3
df['events_6h'] = c6

print(f"    events_1h corr with duration: "
      f"{df[['events_1h','duration_hours']].corr().iloc[0,1]:.3f}")
print(f"    events_3h corr with duration: "
      f"{df[['events_3h','duration_hours']].corr().iloc[0,1]:.3f}")
print(f"    events_6h corr with duration: "
      f"{df[['events_6h','duration_hours']].corr().iloc[0,1]:.3f}")


# ─────────────────────────────────────────────────
# 3. CHRONOLOGICAL SPLIT
# ─────────────────────────────────────────────────
print("\n[3] Chronological 80/20 split...")

split_idx = int(len(df) * TRAIN_RATIO)
train_df  = df.iloc[:split_idx].copy().reset_index(drop=True)
test_df   = df.iloc[split_idx:].copy().reset_index(drop=True)

print(f"    Train : {len(train_df)}  "
      f"({train_df['start_datetime'].min().date()} → "
      f"{train_df['start_datetime'].max().date()})")
print(f"    Test  : {len(test_df)}  "
      f"({test_df['start_datetime'].min().date()} → "
      f"{test_df['start_datetime'].max().date()})")


# ─────────────────────────────────────────────────
# 4. SPATIAL CLUSTERING (train-only fit)
# ─────────────────────────────────────────────────
print(f"\n[4] K-Means spatial clustering ({N_SPATIAL_ZONES} micro-zones)...")

for d in [train_df, test_df]:
    d['latitude']  = d['latitude'].fillna(BENGALURU_LAT)
    d['longitude'] = d['longitude'].fillna(BENGALURU_LNG)

kmeans = KMeans(n_clusters=N_SPATIAL_ZONES, random_state=RANDOM_SEED, n_init=10)
train_df['micro_zone'] = kmeans.fit_predict(
    train_df[['latitude', 'longitude']]
).astype(str)
test_df['micro_zone'] = kmeans.predict(
    test_df[['latitude', 'longitude']]
).astype(str)

print(f"    Clusterer trained on training data only — no spatial leakage.")


# ─────────────────────────────────────────────────
# 5. RISK MAP COMPUTATION (train-only)
# ─────────────────────────────────────────────────
print("\n[5] Computing risk maps (train only)...")

GLOBAL_MEDIAN    = train_df['duration_hours'].median()
GLOBAL_FALLBACK  = np.log1p(GLOBAL_MEDIAN)

corridor_risk_map = np.log1p(
    train_df.groupby('corridor')['duration_hours'].median()
).to_dict()

cause_risk_map = np.log1p(
    train_df.groupby('event_cause')['duration_hours'].median()
).to_dict()

corridor_density_map = (
    train_df['corridor'].value_counts() /
    train_df['corridor'].value_counts().max()
).to_dict()

# Junction risk: vehicle_breakdown only
# Verified: water_logging (0 juncs), construction (1), accident (0)
# survive any threshold — no signal. VB has 49 valid junctions.
vb_train        = train_df[train_df['event_cause'] == 'vehicle_breakdown'].copy()
vb_junc_counts  = vb_train['junction'].value_counts()
vb_valid_juncs  = set(vb_junc_counts[vb_junc_counts >= VB_JUNC_FREQ].index)
vb_junc_risk_map = np.log1p(
    vb_train[vb_train['junction'].isin(vb_valid_juncs)]
    .groupby('junction')['duration_hours'].median()
).to_dict()

print(f"    Corridors : {len(corridor_risk_map)}")
print(f"    Causes    : {len(cause_risk_map)}")
print(f"    VB junctions : {len(vb_junc_risk_map)}")


# ─────────────────────────────────────────────────
# 6. FEATURE ENGINEERING
# ─────────────────────────────────────────────────
print("\n[6] Feature engineering...")

def consolidate_vehicle(val: str) -> str:
    """
    Domain-grounded grouping by operational impact class.
    BMTC/KSRTC buses: fixed routes → prolonged secondary congestion.
    Heavy commercial: hardest to tow → longest clearance.
    """
    v = str(val).lower().strip()
    if any(x in v for x in ['bmtc', 'ksrtc', 'private_bus', 'bus']):
        return 'bus'
    if any(x in v for x in ['heavy', 'truck', 'lorry']):
        return 'heavy_commercial'
    if any(x in v for x in ['lcv', 'loader', 'mini']):
        return 'lcv'
    if any(x in v for x in ['car', 'auto', 'taxi', 'two', 'bike']):
        return 'private_vehicle'
    return 'Unknown'


def engineer(data: pd.DataFrame,
             corr_risk: dict,
             cause_risk: dict,
             corr_density: dict,
             vb_junc_risk: dict,
             fallback: float) -> pd.DataFrame:
    d = data.copy()

    # Vehicle type consolidation
    d['veh_grouped'] = d['veh_type'].fillna('Unknown').apply(consolidate_vehicle)

    # Junction risk: VB only, numerical — avoids 67.5% null + cardinality
    d['vb_junction_risk'] = np.where(
        d['event_cause'] == 'vehicle_breakdown',
        d['junction'].map(vb_junc_risk).fillna(fallback),
        0.0
    )

    # Time features
    d['hour']         = d['start_datetime'].dt.hour
    d['day_of_week']  = d['start_datetime'].dt.dayofweek
    d['month']        = d['start_datetime'].dt.month
    d['day_of_month'] = d['start_datetime'].dt.day
    d['week_of_year'] = d['start_datetime'].dt.isocalendar().week.astype(int)
    d['is_weekend']   = (d['day_of_week'] >= 5).astype(int)

    # Data-grounded time flags
    d['is_night']   = d['hour'].isin(NIGHT_HOURS).astype(int)
    d['is_predawn'] = d['hour'].isin(PREDAWN_HOURS).astype(int)

    # Cyclical encodings — avoid 23→0 discontinuity in tree splits
    d['hour_sin']  = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos']  = np.cos(2 * np.pi * d['hour'] / 24)
    d['dow_sin']   = np.sin(2 * np.pi * d['day_of_week'] / 7)
    d['dow_cos']   = np.cos(2 * np.pi * d['day_of_week'] / 7)
    d['month_sin'] = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos'] = np.cos(2 * np.pi * d['month'] / 12)

    # Target-encoded risk scores (train-only maps, zero leakage)
    d['corridor_risk']    = d['corridor'].map(corr_risk).fillna(fallback)
    d['cause_risk']       = d['event_cause'].map(cause_risk).fillna(fallback)
    d['corridor_density'] = d['corridor'].map(corr_density).fillna(0.0)

    # Interaction features
    d['cause_closure']  = (
        d['event_cause'].fillna('Unknown').astype(str) + '_' +
        d['requires_road_closure'].astype(str)
    )
    d['cause_priority'] = (
        d['event_cause'].fillna('Unknown').astype(str) + '_' +
        d['priority'].fillna('Unknown').astype(str)
    )
    # Multiplicative signal: high-risk cause + closure compounds duration
    d['risk_x_closure'] = d['cause_risk'] * (
        1.0 + d['requires_road_closure'].astype(int) * 0.68
    )

    return d


train_df = engineer(train_df, corridor_risk_map, cause_risk_map,
                    corridor_density_map, vb_junc_risk_map, GLOBAL_FALLBACK)
test_df  = engineer(test_df,  corridor_risk_map, cause_risk_map,
                    corridor_density_map, vb_junc_risk_map, GLOBAL_FALLBACK)

# ── Categorical spec ─────────────────────────────
# police_station : 0% null, 54 unique, all survive >=8 freq threshold.
#   HAL Old Airport (11.8h median) vs Jeevanbheemanagar (0.5h) — real
#   location × infrastructure signal. Variance explained (4.2%) is
#   additive to corridor (7.1%), not duplicative.
# junction       : included as numerical vb_junction_risk, not raw cat.
# micro_zone     : K-Means cluster, always populated.
CAT_FEATURES = [
    'event_type',
    'event_cause',
    'corridor',
    'priority',
    'requires_road_closure',
    'veh_grouped',
    'zone',
    'police_station',
    'micro_zone',
    'cause_closure',
    'cause_priority',
]

NUM_FEATURES = [
    'hour', 'day_of_week', 'month', 'day_of_month', 'week_of_year',
    'is_weekend', 'is_night', 'is_predawn',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'corridor_risk', 'cause_risk', 'corridor_density',
    'risk_x_closure', 'vb_junction_risk',
    'events_1h', 'events_3h', 'events_6h',
]

FEATURES = CAT_FEATURES + NUM_FEATURES

# Frequency clipping — vocabulary from training only, applied to both
for col in CAT_FEATURES:
    train_df[col] = train_df[col].fillna('Unknown').astype(str)
    vc    = train_df[col].value_counts()
    rare  = set(vc[vc < MIN_CAT_FREQ].index)
    known = set(vc[vc >= MIN_CAT_FREQ].index)

    train_df.loc[train_df[col].isin(rare), col] = 'Other'
    train_df[col] = train_df[col].astype('category')

    test_df[col] = test_df[col].fillna('Unknown').astype(str)
    test_df.loc[~test_df[col].isin(known), col] = 'Other'
    test_df[col]  = test_df[col].astype('category')

TARGET      = 'duration_hours'
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]
y_train_log      = np.log1p(y_train)

print(f"    Total features : {len(FEATURES)}  "
      f"({len(CAT_FEATURES)} categorical, {len(NUM_FEATURES)} numerical)")


# ─────────────────────────────────────────────────
# 7. METRICS
# ─────────────────────────────────────────────────
def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric MAPE — robust to near-zero values (31% of test < 30 min)."""
    denom = np.abs(y_true) + np.abs(y_pred) + 1e-8
    return float(100 * np.mean(2 * np.abs(y_pred - y_true) / denom))

def pi_coverage(y_true: np.ndarray, lo: np.ndarray, hi: np.ndarray) -> float:
    """% of actuals inside the 10–90 prediction interval."""
    return float(np.mean((y_true >= lo) & (y_true <= hi)) * 100)

def assign_bucket(h: float) -> str:
    for i in range(len(BUCKET_BOUNDS) - 1):
        if h < BUCKET_BOUNDS[i + 1]:
            return BUCKET_LABELS[i]
    return BUCKET_LABELS[-1]

def bucket_array(arr: np.ndarray) -> np.ndarray:
    return np.array([assign_bucket(h) for h in arr])


# ─────────────────────────────────────────────────
# 8. OPTUNA SEARCH
#    Includes reg_alpha, reg_lambda, min_split_gain
#    Early stopping inside CV — clean trial scores
# ─────────────────────────────────────────────────
print(f"\n[7] Optuna search ({OPTUNA_TRIALS} trials, "
      f"{CV_SPLITS}-fold TimeSeriesCV)...")

def objective(trial: optuna.Trial) -> float:
    params = {
        'objective':         'quantile',
        'alpha':             0.5,
        'metric':            'quantile',
        'verbosity':         -1,
        'random_state':      RANDOM_SEED,
        'n_estimators':      1000,
        'bagging_freq':      1,
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.08, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 90),
        'min_child_samples': trial.suggest_int('min_child_samples', 15, 60),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.5),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.5),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 0.5),
    }

    tscv   = TimeSeriesSplit(n_splits=CV_SPLITS)
    scores = []

    for tr_idx, val_idx in tscv.split(X_train):
        X_tr,  X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr,  y_val = y_train_log.iloc[tr_idx], y_train_log.iloc[val_idx]

        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(-1),
            ]
        )
        preds   = np.expm1(m.predict(X_val))
        actuals = np.expm1(y_val.values)
        scores.append(smape(actuals, preds))

    return float(np.mean(scores))


study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
best_params = study.best_params

print(f"    Best CV SMAPE : {study.best_value:.2f}%")
print(f"    Best params   : {best_params}")


# ─────────────────────────────────────────────────
# 9. TRAIN FINAL QUANTILE MODELS
# ─────────────────────────────────────────────────
print("\n[8] Training final quantile models (q=0.10 / 0.50 / 0.90)...")

final_params = {
    'objective':    'quantile',
    'metric':       'quantile',
    'verbosity':    -1,
    'random_state': RANDOM_SEED,
    'n_estimators': 3000,
    'bagging_freq': 1,
    **best_params,
}

es_cut      = int(len(X_train) * 0.90)
X_fit, X_es = X_train.iloc[:es_cut], X_train.iloc[es_cut:]
y_fit, y_es = y_train_log.iloc[:es_cut], y_train_log.iloc[es_cut:]

es_cb = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]

models = {}
for alpha, name in [(0.10, 'lower'), (0.50, 'median'), (0.90, 'upper')]:
    m = lgb.LGBMRegressor(**{**final_params, 'alpha': alpha})
    m.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], callbacks=es_cb)
    models[name] = m
    print(f"    {name:6s} (q={alpha}) : {m.best_iteration_:>4d} trees")


# ─────────────────────────────────────────────────
# 10. PREDICTIONS + PROTECTION
# ─────────────────────────────────────────────────
p_lo  = np.expm1(models['lower'].predict(X_test))
p_med = np.expm1(models['median'].predict(X_test))
p_hi  = np.expm1(models['upper'].predict(X_test))

# Quantile crossing protection
p_lo  = np.minimum(p_lo,  p_med)
p_hi  = np.maximum(p_hi,  p_med)
# Hard floor
p_lo  = np.maximum(p_lo,  0.0)
p_med = np.maximum(p_med, 0.0)
p_hi  = np.maximum(p_hi,  0.0)


# ─────────────────────────────────────────────────
# 11. SOFT SEVERITY CLASSIFICATION
#     Asymmetric — p50-upward focus (dispatchers care about worst case)
# ─────────────────────────────────────────────────
def soft_classify(lo: float, med: float, hi: float) -> dict:
    """
    Hybrid confidence system.

    Uses BOTH:
    1. Bucket span
    2. Relative uncertainty width

    This avoids the current problem where almost every prediction
    becomes "Low Confidence" simply because it crosses a boundary.
    """

    b_lo  = assign_bucket(lo)
    b_med = assign_bucket(med)
    b_hi  = assign_bucket(hi)

    idx_lo  = BUCKET_LABELS.index(b_lo)
    idx_med = BUCKET_LABELS.index(b_med)
    idx_hi  = BUCKET_LABELS.index(b_hi)

    full_span = idx_hi - idx_lo

    # Relative uncertainty
    baseline = max(med, 0.25)
    relative_width = (hi - lo) / baseline

    # Confidence Rules
    if full_span == 0 and relative_width < 1.5:
        confidence = "High"

    elif full_span <= 1 and relative_width < 4.0:
        confidence = "Moderate"

    else:
        confidence = "Low"

    # Alternate severity only if uncertainty crosses buckets
    alt = b_hi if full_span > 0 else None

    # Confidence score (0-100)
    score = 100 - min(relative_width * 20, 80)
    score = max(20, round(score))

    return {
        'severity': b_med,
        'alt': alt,
        'confidence': confidence,
        'confidence_score': score,
        'idx': idx_med,
        'band_width_buckets': full_span,
        'relative_width': round(relative_width, 2)
    }


classifications = [soft_classify(l, m, h) for l, m, h in zip(p_lo, p_med, p_hi)]
pred_sev  = np.array([c['severity']           for c in classifications])
pred_conf = np.array([c['confidence']         for c in classifications])
pred_idx  = np.array([c['idx']                for c in classifications])
pred_bwb  = np.array([c['band_width_buckets'] for c in classifications])

true_labels = bucket_array(y_test.values)
true_idx    = np.array([BUCKET_LABELS.index(b) for b in true_labels])


# ─────────────────────────────────────────────────
# 12. URGENCY SCORE
#     Composite 0–100, percentage-based, 4 tiers
#     No hardcoded manpower numbers
# ─────────────────────────────────────────────────
def compute_urgency(p50: float, p90: float,
                    bkt_idx: int,
                    closure: bool,
                    cause_risk_val: float,
                    corr_density_val: float) -> dict:
    """
    Four components (weights sum to 100):
      Severity tier     40 pts — which operational bucket
      Spread risk       25 pts — how bad is p90 relative to bucket ceiling
      Recurrence        20 pts — corridor incident density (0–1 normalised)
      Cause risk        15 pts — intrinsic duration risk of this cause type
    """
    sev_pts = [10, 30, 60, 90][bkt_idx]

    ceiling  = BUCKET_BOUNDS[min(bkt_idx + 1, len(BUCKET_BOUNDS) - 1)]
    spread_r = min(p90 / max(ceiling, 0.01), 2.0)
    spread   = spread_r * 12.5
    if closure:
        spread = min(spread * 1.3, 25)

    recurrence = corr_density_val * 20

    cause_n = min(cause_risk_val / 4.28, 1.0)
    cause_s = cause_n * 15

    score = min(max(sev_pts + spread + recurrence + cause_s, 0), 100)

    tier = 'LOW'
    for t, threshold in URGENCY_TIERS.items():
        if score >= threshold:
            tier = t
            break

    return {
        'score':          round(score, 1),
        'tier':           tier,
        'severity_pts':   round(sev_pts, 1),
        'spread_pts':     round(spread, 1),
        'recurrence_pts': round(recurrence, 1),
        'cause_pts':      round(cause_s, 1),
    }


urgency_results = [
    compute_urgency(
        p_med[i], p_hi[i],
        pred_idx[i],
        bool(test_df['requires_road_closure'].iloc[i]),
        test_df['cause_risk'].iloc[i],
        test_df['corridor_density'].iloc[i],
    )
    for i in range(len(test_df))
]
urgency_scores = np.array([u['score'] for u in urgency_results])
urgency_tiers  = np.array([u['tier']  for u in urgency_results])


# ─────────────────────────────────────────────────
# 13. EVALUATION
# ─────────────────────────────────────────────────
print("\n[9] Evaluation on unseen future data...")

mae      = mean_absolute_error(y_test, p_med)
smape_v  = smape(y_test.values, p_med)
cov      = pi_coverage(y_test.values, p_lo, p_hi)
bkt_acc  = (pred_sev == true_labels).mean() * 100
macro_f1 = f1_score(true_idx, pred_idx, average='macro')    * 100
wt_f1    = f1_score(true_idx, pred_idx, average='weighted') * 100

# Confidence tier breakdown
hc_mask  = pred_conf == 'High'
mod_mask = pred_conf == 'Moderate'
lo_mask  = pred_conf == 'Low'

hc_acc  = (pred_sev[hc_mask]  == true_labels[hc_mask]).mean()  * 100 \
          if hc_mask.sum()  > 0 else float('nan')
mod_acc = (pred_sev[mod_mask] == true_labels[mod_mask]).mean() * 100 \
          if mod_mask.sum() > 0 else float('nan')
lo_acc  = (pred_sev[lo_mask]  == true_labels[lo_mask]).mean()  * 100 \
          if lo_mask.sum()  > 0 else float('nan')

hc_pct  = hc_mask.mean()  * 100
mod_pct = mod_mask.mean() * 100
lo_pct  = lo_mask.mean()  * 100

# Naive baseline (cause-majority)
naive_dur = test_df['event_cause'].astype(str).map(
    train_df.groupby('event_cause')['duration_hours'].median().to_dict()
).fillna(GLOBAL_MEDIAN).values.astype(float)

naive_bkt_src = train_df.copy()
naive_bkt_src['bkt'] = naive_bkt_src['duration_hours'].apply(assign_bucket)
naive_bkt_map = naive_bkt_src.groupby('event_cause')['bkt'].agg(
    lambda x: x.value_counts().index[0]).to_dict()
naive_bkt  = test_df['event_cause'].astype(str).map(naive_bkt_map).fillna('Short').values
naive_bidx = np.array([BUCKET_LABELS.index(b) for b in naive_bkt])
naive_bacc = (naive_bkt == true_labels).mean() * 100
naive_mf1  = f1_score(true_idx, naive_bidx, average='macro') * 100

print()
print("  ┌──────────────────────────────────────────────────────────┐")
print("  │                EVALUATION RESULTS — V7                    │")
print("  ├────────────────────────────────┬───────────┬─────────────┤")
print("  │ Metric                         │ CitySync  │    Naive    │")
print("  ├────────────────────────────────┼───────────┼─────────────┤")
print(f"  │ MAE (hours)                    │ {mae:>7.3f}   │ {mean_absolute_error(y_test,naive_dur):>7.3f}     │")
print(f"  │ SMAPE                          │ {smape_v:>6.2f}%   │ {smape(y_test.values,naive_dur):>6.1f}%     │")
print(f"  │ Coverage (10–90 PI)            │ {cov:>6.2f}%   │      —      │")
print("  ├────────────────────────────────┼───────────┼─────────────┤")
print(f"  │ Bucket Accuracy                │ {bkt_acc:>6.2f}%   │ {naive_bacc:>6.1f}%     │")
print(f"  │ Macro F1                       │ {macro_f1:>6.2f}%   │ {naive_mf1:>6.1f}%     │")
print(f"  │ Weighted F1                    │ {wt_f1:>6.2f}%   │      —      │")
print("  ├────────────────────────────────┼───────────┼─────────────┤")
print(f"  │ High-conf   accuracy / share   │ {hc_acc:>5.1f}%  /  {hc_pct:>4.1f}%  │      —      │")
print(f"  │ Moderate-conf accuracy / share │ {mod_acc:>5.1f}%  /  {mod_pct:>4.1f}%  │      —      │")
print(f"  │ Low-conf    accuracy / share   │ {lo_acc:>5.1f}%  /  {lo_pct:>4.1f}%  │      —      │")
print("  └────────────────────────────────┴───────────┴─────────────┘")

print("\n  Per-cause breakdown:")
print(f"  {'Cause':<22} {'n':>4}  {'SMAPE':>7}  {'BktAcc':>7}  "
      f"{'Cov':>6}  {'AvgUrg':>7}  {'HiConf%':>8}")
print(f"  {'─'*22} {'─'*4}  {'─'*7}  {'─'*7}  {'─'*6}  {'─'*7}  {'─'*8}")
for cause in sorted(test_df['event_cause'].unique()):
    mask = test_df['event_cause'].values == cause
    if mask.sum() < 3:
        continue
    s   = smape(y_test.values[mask], p_med[mask])
    ba  = (pred_sev[mask] == true_labels[mask]).mean() * 100
    cv  = pi_coverage(y_test.values[mask], p_lo[mask], p_hi[mask])
    au  = urgency_scores[mask].mean()
    hcp = (pred_conf[mask] == 'High').mean() * 100
    print(f"  {cause:<22} {mask.sum():>4}  {s:>6.1f}%  {ba:>6.1f}%  "
          f"{cv:>5.1f}%  {au:>7.1f}  {hcp:>7.1f}%")

print("\n  Confidence distribution:")
for conf, mask in [('High', hc_mask), ('Moderate', mod_mask), ('Low', lo_mask)]:
    n = mask.sum()
    print(f"    {conf:<10} {n:>4}  ({n/len(pred_conf)*100:.1f}%)")

print("\n  Urgency tier distribution:")
for tier in ['CRITICAL', 'HIGH', 'MODERATE', 'LOW']:
    n = (urgency_tiers == tier).sum()
    print(f"    {tier:<10} {n:>4}  ({n/len(urgency_tiers)*100:.1f}%)")

print(f"\n  Beats naive baseline (Macro F1): "
      f"{'YES ✓' if macro_f1 > naive_mf1 else 'NO ✗'}")


# ─────────────────────────────────────────────────
# 14. FIGURES
# ─────────────────────────────────────────────────
print(f"\n[10] Generating figures...")

# — Feature Importance ——————————————————————————
fi = (
    pd.Series(models['median'].feature_importances_, index=FEATURES)
    .sort_values(ascending=True).tail(18)
)
fig, ax = plt.subplots(figsize=(10, 7))
colors = [ACCENT if v >= fi.values[-5] else BORDER for v in fi.values]
ax.barh(fi.index, fi.values, color=colors, edgecolor='none', height=0.7)
ax.set_xlabel('Feature Importance (Gain)', color=MUTED, fontsize=9)
ax.set_title('CitySync V7 — Feature Importance (Median Model)',
             color=TEXT, fontsize=11, pad=12, loc='left')
ax.tick_params(colors=MUTED, labelsize=8)
for sp in ax.spines.values():
    sp.set_edgecolor(BORDER)
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(SURFACE)
plt.tight_layout()
plt.savefig(FIG_IMPORTANCE, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()
print(f"    Saved: {FIG_IMPORTANCE}")

# — Confusion Matrix ————————————————————————————
cm      = confusion_matrix(true_idx, pred_idx)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(DARK_BG)
for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ['Confusion Matrix (counts)', 'Row-normalised'],
    ['d', '.2f']
):
    ax.set_facecolor(SURFACE)
    ax.imshow(data, cmap='Blues', aspect='auto', vmin=0, vmax=data.max())
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(BUCKET_LABELS, color=MUTED, fontsize=9)
    ax.set_yticklabels(BUCKET_LABELS, color=MUTED, fontsize=9)
    ax.set_xlabel('Predicted', color=MUTED, fontsize=9)
    ax.set_ylabel('Actual',    color=MUTED, fontsize=9)
    ax.set_title(title, color=TEXT, fontsize=10, pad=8)
    for i in range(4):
        for j in range(4):
            v = data[i, j]
            ax.text(j, i, f'{v:{fmt}}', ha='center', va='center',
                    color=TEXT if v < data.max() * 0.6 else DARK_BG,
                    fontsize=9, fontweight='bold')
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
plt.suptitle('Severity Classification Performance', color=TEXT,
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIG_CONFUSION, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()
print(f"    Saved: {FIG_CONFUSION}")

# — Calibration ————————————————————————————————
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(DARK_BG)

ax = axes[0]
ax.set_facecolor(SURFACE)
for i, (label, color) in enumerate(zip(BUCKET_LABELS, BUCKET_COLOURS)):
    mb = true_idx == i
    ax.scatter(y_test.values[mb], p_med[mb], color=color, alpha=0.5,
               s=16, label=label, edgecolors='none')
mv = min(float(y_test.max()), 72.0)
ax.plot([0, mv], [0, mv], '--', color=MUTED, lw=1, alpha=0.6)
ax.set_xlabel('Actual Duration (hrs)', color=MUTED, fontsize=9)
ax.set_ylabel('Predicted Median (hrs)', color=MUTED, fontsize=9)
ax.set_title('Predicted vs Actual', color=TEXT, fontsize=10, pad=8)
ax.tick_params(colors=MUTED, labelsize=8)
ax.legend(fontsize=8, facecolor=SURFACE, labelcolor=TEXT, edgecolor=BORDER)
for sp in ax.spines.values():
    sp.set_edgecolor(BORDER)

ax2 = axes[1]
ax2.set_facecolor(SURFACE)
cov_bkt = []
for i, label in enumerate(BUCKET_LABELS):
    mb = true_idx == i
    cov_bkt.append(
        pi_coverage(y_test.values[mb], p_lo[mb], p_hi[mb])
        if mb.sum() > 0 else 0.0
    )
bars = ax2.bar(BUCKET_LABELS, cov_bkt, color=BUCKET_COLOURS,
               edgecolor='none', alpha=0.85)
ax2.axhline(80, color=ACCENT, lw=1.2, linestyle='--',
            alpha=0.7, label='80% target')
ax2.axhline(cov, color=MUTED, lw=1, linestyle=':',
            alpha=0.5, label=f'Overall {cov:.1f}%')
for bar, val in zip(bars, cov_bkt):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 1.5,
             f'{val:.1f}%', ha='center', color=TEXT,
             fontsize=9, fontweight='bold')
ax2.set_ylim(0, 115)
ax2.set_xlabel('Severity Bucket', color=MUTED, fontsize=9)
ax2.set_ylabel('Coverage (%)', color=MUTED, fontsize=9)
ax2.set_title('Band Coverage by Severity', color=TEXT, fontsize=10, pad=8)
ax2.tick_params(colors=MUTED, labelsize=8)
ax2.legend(fontsize=8, facecolor=SURFACE, labelcolor=TEXT, edgecolor=BORDER)
for sp in ax2.spines.values():
    sp.set_edgecolor(BORDER)

plt.tight_layout()
plt.savefig(FIG_CALIBRATION, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()
print(f"    Saved: {FIG_CALIBRATION}")


# ─────────────────────────────────────────────────
# 15. SAMPLE OPERATIONAL OUTPUTS
# ─────────────────────────────────────────────────
print("\n[11] Sample operational outputs:")
for i, idx in enumerate(test_df.sample(6, random_state=99).index.tolist()):
    pos = test_df.index.get_loc(idx)
    row = test_df.iloc[pos]
    u   = urgency_results[pos]
    c   = classifications[pos]
    alt = f' → {c["alt"]}?' if c['alt'] else ''
    bwb = c['band_width_buckets']
    print(f"\n  [{i+1}] {row['event_cause']}  |  {row['corridor']}")
    print(f"      Band      : {p_lo[pos]:.1f} – {p_med[pos]:.2f} – {p_hi[pos]:.1f} hrs")
    print(f"      Actual    : {y_test.iloc[pos]:.2f} hrs")
    print(
    f"      Severity  : {c['severity']}{alt}  "
    f"[{c['confidence']} confidence "
    f"({c['confidence_score']}/100), span={bwb}]"
)
    print(f"      Urgency   : {u['score']:.0f}/100  →  {u['tier']}")


# ─────────────────────────────────────────────────
# 16. EXPORT BUNDLE
# ─────────────────────────────────────────────────
print(f"\n[12] Exporting bundle → {OUTPUT_PKL}")

bundle = {
    'models':               models,
    'features':             FEATURES,
    'cat_features':         CAT_FEATURES,
    'num_features':         NUM_FEATURES,
    'corridor_risk_map':    corridor_risk_map,
    'cause_risk_map':       cause_risk_map,
    'corridor_density_map': corridor_density_map,
    'vb_junc_risk_map':     vb_junc_risk_map,
    'global_fallback':      GLOBAL_FALLBACK,
    'global_median':        GLOBAL_MEDIAN,
    'kmeans':               kmeans,
    'known_categories':     {col: list(train_df[col].cat.categories)
                             for col in CAT_FEATURES},
    'bucket_bounds':        BUCKET_BOUNDS,
    'bucket_labels':        BUCKET_LABELS,
    'urgency_tiers':        URGENCY_TIERS,
    'night_hours':          NIGHT_HOURS,
    'predawn_hours':        PREDAWN_HOURS,
    'evaluation': {
        'mae':          round(mae, 3),
        'smape':        round(smape_v, 2),
        'coverage':     round(cov, 2),
        'bucket_acc':   round(bkt_acc, 2),
        'macro_f1':     round(macro_f1, 2),
        'weighted_f1':  round(wt_f1, 2),
        'hc_acc':       round(hc_acc, 2) if not np.isnan(hc_acc) else None,
        'hc_pct':       round(hc_pct, 2),
        'mod_acc':      round(mod_acc, 2) if not np.isnan(mod_acc) else None,
        'mod_pct':      round(mod_pct, 2),
        'lo_acc':       round(lo_acc, 2) if not np.isnan(lo_acc) else None,
        'lo_pct':       round(lo_pct, 2),
        'naive_smape':  round(smape(y_test.values, naive_dur), 2),
        'naive_mf1':    round(naive_mf1, 2),
        'train_size':   len(X_train),
        'test_size':    len(X_test),
    },
}

joblib.dump(bundle, OUTPUT_PKL)

print(f"    3 quantile models  |  {len(FEATURES)} features  |  "
      f"K-Means clusterer  |  5 risk maps")
print()
print("=" * 60)
print(f"  V7 COMPLETE — MAE {mae:.3f}h  |  Coverage {cov:.1f}%  |  "
      f"MacroF1 {macro_f1:.1f}%")
print("=" * 60)

  CITYSYNC COMMAND — V7 FINAL MODEL

[1] Loading and resolving target variable...
    Records : 2917
    Range   : 0.00 – 71.87 hrs
    Median  : 0.88 hrs  (~53 min)
    Dates   : 2023-11-09 → 2024-04-08

[2] Computing leakage-free rolling corridor counts...
    events_1h corr with duration: 0.144
    events_3h corr with duration: 0.155
    events_6h corr with duration: 0.132

[3] Chronological 80/20 split...
    Train : 2333  (2023-11-09 → 2024-03-14)
    Test  : 584  (2024-03-14 → 2024-04-08)

[4] K-Means spatial clustering (25 micro-zones)...
    Clusterer trained on training data only — no spatial leakage.

[5] Computing risk maps (train only)...
    Corridors : 22
    Causes    : 14
    VB junctions : 49

[6] Feature engineering...
    Total features : 33  (11 categorical, 22 numerical)

[7] Optuna search (60 trials, 4-fold TimeSeriesCV)...
    Best CV SMAPE : 76.15%
    Best params   : {'learning_rate': 0.03268424672361593, 'num_leaves': 50, 'min_child_samples': 26, 'bagging_frac